# Arpa-150M — Pré-treino A100 (turnkey)

**Runtime > Alterar tipo de runtime > GPU > A100** antes de rodar.

Depois, **Ambiente de execução > Executar tudo**. O notebook é idempotente:
tokenizer e bins são gerados **uma vez** e salvos no teu Drive; os checkpoints
vão direto pro Drive (symlink). Se o Colab cair, basta **Executar tudo** de novo
— ele restaura tudo do Drive e retoma o treino exatamente de onde parou.

Tudo fica em `MyDrive/arpa150m/`.

## 1. Checar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA:', torch.cuda.is_available(), '| BF16:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 2. Dependências

In [ ]:
!pip install -q torch transformers datasets tokenizers

## 3. Montar Drive + pasta de persistência

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/arpa150m'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Persistência em:', DRIVE_ROOT)

## 4. Clonar o repo (sempre a versão mais recente do GitHub)

In [ ]:
REPO = '/content/ResNet-Psi'
if os.path.exists(REPO):
    !cd {REPO} && git pull --ff-only
else:
    !git clone https://github.com/Lucas-404/ResNet-Psi.git {REPO}
%cd {REPO}
!git log -1 --oneline

## 5. Tokenizer 64K (gera 1x ~30min, depois restaura do Drive)

In [ ]:
import shutil
TOK = 'tokenizer-arpa-64k-clean'
drive_tok = f'{DRIVE_ROOT}/{TOK}'
if os.path.exists(os.path.join(drive_tok, 'tokenizer.json')):
    print('Tokenizer encontrado no Drive — restaurando')
    if os.path.exists(TOK):
        shutil.rmtree(TOK)
    shutil.copytree(drive_tok, TOK)
else:
    print('Treinando tokenizer 64K (uma vez)...')
    !python arpa/train_tokenizer.py
    shutil.copytree(TOK, drive_tok)
    print('Salvo no Drive')
assert os.path.exists(os.path.join(TOK, 'tokenizer.json')), 'tokenizer falhou'
print('tokenizer OK')

## 6. Bins de pré-treino (gera 1x ~6.6GB, depois restaura do Drive)

In [ ]:
BIN_DIR = 'data/arpa150m'
drive_bin = f'{DRIVE_ROOT}/data'
files = ('train_tokens.bin', 'val_tokens.bin')
if os.path.exists(os.path.join(drive_bin, 'train_tokens.bin')):
    print('Bins encontrados no Drive — restaurando')
    os.makedirs(BIN_DIR, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(drive_bin, f), os.path.join(BIN_DIR, f))
else:
    print('Gerando bins (uma vez, streaming do mix real)...')
    !python arpa/prepare_data.py --train-tokens 3.3e9
    os.makedirs(drive_bin, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(BIN_DIR, f), os.path.join(drive_bin, f))
    print('Salvo no Drive')
assert os.path.exists(os.path.join(BIN_DIR, 'train_tokens.bin')), 'bins falharam'
print('bins OK')

## 7. Checkpoints vão direto pro Drive (sobrevivem a desconexão)

In [ ]:
ckpt_drive = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(ckpt_drive, exist_ok=True)
link = 'checkpoints-arpa150m'
if os.path.islink(link) or os.path.exists(link):
    if not os.path.islink(link):
        shutil.rmtree(link)
if not os.path.exists(link):
    os.symlink(ckpt_drive, link)
print(link, '->', os.path.realpath(link))

## 8. Pré-treino A100 (~12-18h para 3.2B tokens)

Auto-detecta checkpoints no Drive: se existirem, **retoma**; senão, começa do zero.
Pode rodar esta célula quantas vezes precisar — sempre continua de onde parou.

In [ ]:
import glob
resume = '--resume latest' if glob.glob('checkpoints-arpa150m/step_*.pt') else ''
print('RETOMANDO treino' if resume else 'COMEÇANDO do zero')
!python arpa/pretrain.py --config a100 {resume}

## 9. (Depois do pré-treino) SFT + chat

Rode estas células só quando o pré-treino terminar (existir `best.pt`).

In [ ]:
# SFT (fine-tune supervisionado nos seeds em sft/)
!python arpa/sft.py --init checkpoints-arpa150m/best.pt --data "sft/*.jsonl"

In [ ]:
# Conversar com o modelo treinado
!python arpa/sample.py --ckpt checkpoints-sft/best.pt --chat